# CYBER SECURITY PER LA SANITA' TRAMITE REINFORCEMENT LEARNING

**DeepGuard Inc.** e' un'azienda che sviluppa soluzioni di sicurezza informatica per il settore sanitario. Su incarico di **GreenGuard Solutions**, l'obbiettivo è valutare se tecniche di **Reinforcement Learning** possano essere usate per addestrare un agente difensivo capace di individuare un attaccante che si muove dentro una rete informatica, prima che questo raggiunga i dati sensibili dei pazienti.

Il campo di prova e' [`gym-idsgame`](https://github.com/Limmen/gym-idsgame), un ambiente Gym che simula in forma semplificata una rete a livelli (Start &rarr; Server &rarr; Data): un attaccante cerca di farsi strada nodo dopo nodo fino al nodo Data, un difensore alza le difese sui nodi per bloccarlo o smascherarlo.

**Obbiettivo del notebook:**

1. Si allena un agente **SARSA** (tabellare) a difendere la rete da un attaccante che si muove in modo casuale (scenario *random attack*).
2. Si allena un agente **Double DQN** (rete neurale in PyTorch) sullo stesso scenario *random attack* e su uno scenario piu' ostile, *maximal attack*, dove l'attaccante sceglie sempre la mossa che sfrutta il punto piu' debole invece di muoversi a caso.
3. Si confrontano i due algoritmi tra loro e con due difensori "ingenui" (uno casuale, uno a regola fissa), per capire se e quando l'apprendimento porta davvero un vantaggio misurabile.

## SEZIONE 1: INSTALLAZIONE E CONFIGURAZIONE DELL'AMBIENTE

Il progetto indica come fonte dell'ambiente il repository GitHub [`gym-idsgame`](https://github.com/Limmen/gym-idsgame) (branch `master`), e come riferimento per la configurazione la guida `Guida_Setup_Ambiente.ipynb` (nella stessa cartella).

Il codice sul branch `master` di GitHub e' stato migrato a **`gymnasium`** (API a 5 valori per `step()`/`reset()`), mentre la guida di riferimento si basa sulla vecchia libreria `gym` (API a 4 valori, tuple annidate). Per questo, dalla guida sono stati ripresi solo gli elementi indipendenti dall'API:
- l'uso della versione **`v21`** dell'ambiente;
- il **monkey-patch** per il bug del bot dell'attaccante casuale (Sezione 2).

Non e' stato invece ripreso il wrapper della guida (`DefenderWrapper`), scritto per la vecchia API `gym`: `DefenderGymWrapper` (Sezione 3) e' scritto da zero per l'API `gymnasium`, e tiene inoltre traccia di qualche informazione in piu' (dimensione dell'osservazione, id del nodo Data, esito vero della partita) usata nelle sezioni successive.

### Le dipendenze

`gym-idsgame` non e' un pacchetto mantenuto in modo continuo: il file `setup.py` pubblicato nel repository dichiara ancora come dipendenza la vecchia libreria `gym`, mentre il codice sorgente attuale (branch `master`) e' stato migrato a **`gymnasium`** e usa la sua API a 5 valori (`obs, reward, terminated, truncated, info`) sia per `step()` che per `reset()`.

In [ ]:
# ============================================================
# INSTALLAZIONE DELL'AMBIENTE DI SIMULAZIONE gym-idsgame
# ============================================================
!pip install -q gymnasium
!pip install -q --no-deps --force-reinstall "git+https://github.com/Limmen/gym-idsgame.git"

In [ ]:
# ============================================================
# IMPORT E VERIFICA DELL'INSTALLAZIONE
# ============================================================
import warnings
warnings.filterwarnings("ignore")  # silenzia i warning di deprecazione di gymnasium sulle vecchie versioni degli ambienti idsgame

import random
import time
from collections import deque, Counter, namedtuple, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

import gymnasium as gym
import gym_idsgame  # l'import da solo registra tutti gli ambienti idsgame-* nel registry di gymnasium

In [ ]:
# ============================================================
# SEED PER RIPRODUCIBILITA'
# ============================================================
# Si fissa il seed su tutti i generatori casuali coinvolti, cosi' che le scelte
# "a caso" dell'agente (quando esplora invece di sfruttare quanto ha gia' imparato)
# siano sempre le stesse a ogni esecuzione.
#
# Nota: questo seed copre SOLO le estrazioni casuali fatte con random/np.random
# (compresa l'esplorazione epsilon-greedy di SARSA e DDQN). Non copre invece le
# estrazioni casuali fatte internamente da gymnasium quando si chiama .sample() su
# uno spazio di azioni (usato per il difensore casuale di confronto): 
# quegli spazi hanno un proprio generatore interno, separato,
# che si inizializza da solo e ignora questo SEED. 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

## SEZIONE 2: COME FUNZIONA gym-idsgame

Prima di scrivere un algoritmo di RL è bene capire bene le regole del gioco.

**La rete.** Viene usata la versione `v21` dell'ambiente: 4 nodi in tutto, `Start` (da cui parte l'attaccante), 2 `Server` intermedi e `Data` (l'obiettivo finale, dove risiedono i dati sensibili). L'attaccante deve muoversi da `Start` a `Data` passando per uno dei due server.
L'attaccante ha due percorsi possibili per arrivare all'obiettivo, non uno solo:

```text
                    Data (nodo 0)
                  obiettivo finale
                   /            \
      Server (nodo 1)        Server (nodo 2)
                   \            /
                    Start (nodo 3)
              (l'attaccante parte da qui)
```

**Chi viene controllato.** Negli scenari `idsgame-random_attack-vN` e `idsgame-maximal_attack-vN` l'agente RL controlla il **difensore**; l'attaccante e' un bot scriptato:
- in `random_attack` sceglie ogni mossa a caso tra quelle legali;
- in `maximal_attack` e' deterministico e attacca sempre il nodo raggiungibile con il valore di attacco piu' alto, usando il tipo di attacco su cui ha gia' fatto piu' progressi (in pratica: concentra lo sforzo dove ha piu' probabilita' di sfondare, invece di esplorare a caso).

**Osservazione.** Per ogni nodo, il difensore vede un numero da 0 a 9 (`max_value=9`) per ciascuno dei suoi attributi.

Gli attributi per nodo sono 5 in totale, ma non sono tutti uguali: si dividono in due gruppi con una logica diversa.

- **4 attributi di difesa "normali"** (tipo 0, 1, 2 o 3): ciascuno corrisponde a uno dei 4 tipi di attacco possibili. Un attacco riesce solo se il valore di attacco dell'attaccante per un certo tipo e' piu' alto del valore di difesa dello stesso identico tipo su quel nodo — sono quattro "braccia di ferro" indipendenti, una per tipo.
- **1 attributo speciale, il livello di "rilevamento"**: non partecipa a nessun duello. Entra in gioco solo dopo che un attacco e' gia' fallito, e decide se il difensore se ne accorge comunque. La probabilita' di essere scoperti e' `livello_di_rilevamento / 10`:

  | Livello di rilevamento | Calcolo | Probabilita' di essere scoperti |
  |---|---|---|
  | 0 (mai rinforzato) | 0/10 | 0% — impossibile |
  | 3 | 3/10 | 30% |
  | 5 | 5/10 | 50% — una moneta lanciata |
  | 9 (il massimo) | 9/10 | 90% |

Il tetto di 9 non e' una regola fissa del gioco in generale, ma il parametro `max_value` della configurazione (`GameConfig`), che per la versione `v21` usata in questo notebook vale appunto 9. 
Il limite sulla libreria viene indicato "aumentando di 1, ma solo se non e' gia' stato raggiunto il massimo". Se il livello e' gia' a 9 e viene rigiocata l'azione di rilevamento sullo stesso nodo, non succede nulla — l'azione viene accettata senza errore, ma non ha piu' alcun effetto.

Con 4 nodi nella rete e 5 attributi per nodo, l'osservazione completa e' quindi una tabella `4 x 5`. Questa tabella viene poi appiattita in un unico elenco di 20 numeri (4 righe x 5 colonne = 20), perche' e' il formato che si aspettano sia la Q-table di SARSA sia la rete neurale di DDQN: nessuna informazione va persa, cambia solo il modo in cui e' organizzata, da griglia a lista.

**Azione.** Ogni azione del difensore corrisponde esattamente a una casella della tabella appena descritta: significa "scegli questo nodo, rinforza questo attributo di un'unita'". Se l'attributo scelto e' l'ultimo (il rilevamento), sale il livello di rilevamento del nodo; per gli altri 4, sale il livello di difesa normale su quell'attributo specifico. Dato che la tabella ha 4 righe (nodi) e 5 colonne (attributi), le azioni possibili in totale sono 20 — tante quante le caselle della tabella, uno-a-uno. Non tutte e 20 sono altrettanto utili: rinforzare un attributo del nodo `Start` non serve a nulla nella pratica, perche' non e' un nodo da proteggere (l'attaccante parte gia' da li', non deve conquistarlo). L'ambiente accetta comunque quell'azione senza dare errore, semplicemente non ha alcun effetto sul gioco. Non si impedisce pero' esplicitamente all'agente di sceglierle: si lascia che impari da solo, giocando, che sono inutili — cosi' il confronto tra le tecniche resta onesto, senza che nessuna riceva un aiuto che le altre non hanno.

**Reward.** A differenza della versione `v0` (che usa una reward sparsa +1/-1/-100), la versione `v21` ha la **reconnaissance abilitata** e usa uno schema di reward piu' articolato (`dense_rewards_v3`). E' stato verificato: un rilevamento vero puo' dare reward `0` al difensore, una violazione vera puo' dare un valore negativo variabile (es. `-26`), non sempre `-1`. Per questo **non** si classifica l'esito di una partita guardando la soglia del reward (sarebbe fragile e specifico di questa versione): si legge invece direttamente lo stato vero del gioco, `env.state.hacked` e `env.state.detected`, che dicono la verita' a prescindere dallo schema di reward in uso — lo si vedra' tra poco nel wrapper (la demo del difensore casuale qui sotto stampa gia' il reward reale a ogni step, prima di classificare l'esito).

### Il monkey-patch per il bug del bot attaccante

Fix: import mancante nei bot agent

C'è un bug nella libreria: i file `random_attack_bot_agent.py` e `attack_maximal_value_bot_agent.py` usano `idsgame_util` senza importarlo, causando un `NameError` al primo `step()`. Lo risolviamo con un monkey-patch: iniettiamo il modulo mancante nello scope dei bot agent.

Questo fix è necessario solo se l'ambiente decide di invocare internamente il bot agent. Nel nostro wrapper passeremo un placeholder (`-1`) come azione attaccante per evitare che il bot venga chiamato, ma il patch resta come rete di sicurezza.

In [ ]:
# ============================================================
# PATCH: import mancante nei bot agent (bug della libreria, non nostro)
# ============================================================
import gym_idsgame.envs.util.idsgame_util as _idsgame_util
import gym_idsgame.agents.bot_agents.random_attack_bot_agent as _random_bot
import gym_idsgame.agents.bot_agents.attack_maximal_value_bot_agent as _maximal_bot

if not hasattr(_random_bot, "idsgame_util"):
    _random_bot.idsgame_util = _idsgame_util
    print("Patch applicata: random_attack_bot_agent")

if not hasattr(_maximal_bot, "idsgame_util"):
    _maximal_bot.idsgame_util = _idsgame_util
    print("Patch applicata: attack_maximal_value_bot_agent")


In [ ]:
# ============================================================
# CREAZIONE DI UN AMBIENTE DI PROVA
# ============================================================
from gym_idsgame.envs import IdsGameRandomAttackV21Env, IdsGameMaximalAttackV21Env
probe_env = IdsGameRandomAttackV21Env()

# Lo spazio delle azioni di gymnasium ha un generatore casuale proprio, separato da
# random/np.random: senza questa riga, ogni .sample() su questo
# spazio darebbe risultati diversi a ogni esecuzione, seed o non seed.
probe_env.defender_action_space.seed(SEED)

In [ ]:
# ============================================================
# VERIFICA
# ============================================================
print("observation_space dichiarato dall'ambiente:", probe_env.observation_space)

probe_env.reset()
obs_attacker, obs_defender = probe_env.get_observation()
print("Forma osservazione ATTACCANTE (dati veri):", obs_attacker.shape, "-> totale valori:", obs_attacker.size)
print("Forma osservazione DIFENSORE (dati veri): ", obs_defender.shape, "-> totale valori:", obs_defender.size)

# Eseguendolo, si vede che observation_space ha forma (1, 5) - 5 valori - che non
# coincide ne' con i 40 valori veri dell'attaccante ne' con i 20 veri del difensore:
# non descrive correttamente nessuno dei due. Per questo DefenderGymWrapper (Sezione 3)
# non legge mai observation_space per calcolare obs_dim: misura la dimensione
# sull'osservazione VERA restituita da get_observation(), non su un attributo dichiarato
# che qui risulta fuorviante.

In [ ]:
# ============================================================
# UNA PARTITA CASUALE PER "VEDERE" IL GIOCO
# ============================================================

# Riporta l'ambiente allo stato iniziale (rete pulita, attaccante nel nodo Start).
probe_env.reset()

done = False    # diventera' True quando la partita finisce (rilevamento/violazione/timeout)
step_n = 0      # conta quanti passi sono stati fatti

print(f"{'step':>4} | {'azione difensore':>18} | {'reward difensore':>16} | {'done':>5}")

# Il limite di 15 passi e' solo di sicurezza per questa demo.
while not done and step_n < 15:
    # Campiona un'azione a caso tra le 20 possibili
    defense_action = probe_env.defender_action_space.sample()

    # step() si aspetta la tupla (azione_attaccante, azione_difensore). Passiamo None al
    # posto dell'azione dell'attaccante perche' in un DefenderEnv l'ambiente la ignora
    # comunque e muove l'attaccante con il proprio bot interno; basta che non sia -1.
    action = (None, defense_action)  # il difensore e' l'unico ruolo controllato dall'esterno

    # Si esegue il passo vero. Si scarta con "_" l'osservazione restituita: anche qui e'
    # quella dell'attaccante, non quella del difensore, e in questa demo non serve.
    # reward e' la coppia (reward_attaccante, reward_difensore).
    _, reward, terminated, truncated, info = probe_env.step(action)

    done = terminated or truncated

    # reward[1] e' il reward del DIFENSORE (reward[0] sarebbe quello dell'attaccante).
    print(f"{step_n:>4} | {defense_action:>18} | {reward[1]:>16} | {str(done):>5}")
    step_n += 1 

# Si classifica l'esito leggendo lo STATO VERO del gioco, non il valore del reward.
if probe_env.state.hacked:
    outcome_text = "l'attaccante ha violato il nodo Data"
elif probe_env.state.detected:
    outcome_text = "il difensore ha rilevato l'attaccante"
else:
    outcome_text = "timeout (o partita interrotta dal limite di sicurezza della demo), nessuno dei due ha davvero vinto"
print(f"\nEsito della partita: {outcome_text} (durata: {step_n} passi)")

## SEZIONE 3: UN WRAPPER UNICO PER GIOCARE DA DIFENSORE

Questo wrapper trasforma l'ambiente multi-agente in un ambiente single-agent compatibile con qualsiasi algoritmo RL standard (SARSA, Q-Learning, DQN, DDQN, ecc.).

Cosa fa:
- Restituisce sempre l'osservazione del difensore, letta direttamente con `env.get_observation()`, invece di affidarsi al valore di ritorno di `reset()`/`step()` — che, come verificato, restituiscono sempre l'osservazione dell'ATTACCANTE, non quella del difensore.
- Espone un'interfaccia identica per qualsiasi ambiente `idsgame-*-vN`: `reset()` restituisce l'osservazione del difensore gia' appiattita e convertita in `np.float32` (molti algoritmi RL lavorano meglio con osservazioni numeriche in formato float), `step(azione)` restituisce `(osservazione, reward_difensore, terminated, truncated, esito)`.
- Calcola una volta sola, alla creazione, alcune informazioni utili piu' avanti (dimensione dell'osservazione, numero di tipi di attacco, id del nodo Data), cosi' il resto del notebook non deve ricavarle ogni volta.
- Classifica esplicitamente l'esito dell'episodio (`detected`, `breached`, `timeout`) leggendo lo stato vero del gioco (`state.hacked`/`state.detected`), non il valore della reward: servira' per tutte le metriche della prossima sezione, invece di dover ricalcolare la stessa logica in ogni punto del notebook.

In [ ]:
# ============================================================
# WRAPPER: DefenderGymWrapper
# ============================================================
class DefenderGymWrapper:
    '''
    Wrapper per un ambiente idsgame-*-vN che espone all'agente solo il punto di vista del difensore, 
    gestendo questi comportamenti:
    - reset()/step() dell'ambiente restituiscono sempre l'osservazione dell'attaccante;
    - env.action_space e' sempre lo spazio azioni dell'attaccante (si usa defender_action_space);
    - su v21 (reward "dense_v3") il valore del reward non basta a dire chi ha vinto: leggiamo
      lo stato vero del gioco (state.hacked / state.detected) invece di una soglia sul reward.
    '''

    def __init__(self, env):
        self.env = env

        # Numero di azioni disponibili per il difensore. 
        # Non si usa mai env.action_space (ATTACCANTE): sempre defender_action_space.
        self.n_actions = self.env.defender_action_space.n
        self.action_space = env.defender_action_space

        # Si chiama la reset() per avere subito un'osservazione vera con cui calcolare quante dimensioni ha (obs_dim):
        # servira' piu' avanti per dimensionare correttamente la rete neurale del DDQN.
        obs = self.reset()
        self.obs_dim = obs.size

        # Id del nodo Data e numero di "tipi" di attacco: serviranno nella Sezione 4
        # per costruire un difensore euristico di confronto (non servono agli agenti RL).
        game_config = self.env.idsgame_config.game_config
        self.num_attack_types = game_config.num_attack_types
        self.data_node_id = game_config.network_config.get_node_id(game_config.network_config.data_pos)

    def _defender_observation(self) -> np.ndarray:
        # get_observation() restituisce SEMPRE la coppia (osservazione_attaccante,
        # osservazione_difensore), a prescindere da cosa restituiscano reset()/step().
        _, defender_obs = self.env.get_observation()
        # .flatten() appiattisce la matrice in un vettore (piu' comodo sia per la chiave
        # della Q-table di SARSA sia per l'input della rete di DDQN).
        # astype(np.float32) uniforma il tipo: la rete neurale si aspetta float, non int.
        return defender_obs.flatten().astype(np.float32)

    def reset(self) -> np.ndarray:
        # reset() viene chiamato solo per il suo effetto collaterale (riportare lo stato del
        # gioco all'inizio).
        self.env.reset()
        return self._defender_observation()

    def step(self, defense_action: int):
        # L'attaccante e' interno all'ambiente (bot scriptato): si passa None al suo posto,
        # cosi' come mostrato nella documentazione della libreria per i ruoli DefenderEnv.
        action = (None, defense_action)
        # Anche qui si scarta con "_" l'osservazione restituita direttamente da step():
        # la osservazione giusta viene ricavata sempre da _defender_observation() sotto.
        _, reward, terminated, truncated, info = self.env.step(action)

        obs = self._defender_observation()

        # reward e' la coppia (reward_attaccante, reward_difensore): teniamo solo la seconda.
        defender_reward = float(reward[1])

        # NOTA SULLA LIBRERIA (verificato leggendo il sorgente di gym-idsgame su GitHub,
        # idsgame_env.py): non fa MAI una vera distinzione tra "partita davvero finita" e
        # "tagliata per timeout" - restituisce sempre terminated == truncated (True, True sia
        # per un timeout sia per una fine partita vera).
        #
        # Si classifica l'esito leggendo lo STATO VERO del gioco, non il valore del reward:
        # sono le stesse due condizioni, mutuamente esclusive, che decidono se l'ambiente
        # imposta terminated/truncated=True (la terza possibilita', il timeout, e' l'unico
        # caso in cui la partita finisce senza che nessuna delle due sia vera).
        outcome = None
        if terminated or truncated:
            if self.env.state.hacked:
                outcome = "breached"
            elif self.env.state.detected:
                outcome = "detected"
            else:
                outcome = "timeout"

        return obs, defender_reward, terminated, truncated, outcome

    def detect_action_for(self, node_id: int) -> int:
        """ 
        Restituisce l'id dell'azione difensiva che aumenta il livello di rilevamento per uno 
        specifico nodo della rete. Nell'ambiente, ogni nodo ha un gruppo di azioni difensive associate. 
        Se esistono num_attack_types tipi di attacco, per ogni nodo ci sono: 
        - una possibile difesa per ciascun tipo di attacco; 
        - una ulteriore azione dedicata al rilevamento. 
        Quindi il numero totale di azioni per nodo è: 
           num_attack_types + 1 
        Per trovare l'id dell'azione di rilevamento di un nodo, si parte dal blocco di azioni 
        associato a quel nodo: 
           node_id * (num_attack_types + 1) 
        e si aggiunge l'indice dell'azione di rilevamento, che nella codifica della libreria è 
        l'ultima azione del blocco:
          num_attack_types 
        Formula finale: 
          node_id * (num_attack_types + 1) + num_attack_types 

        ES PRATICO:
        Supponiamo che ci siano 3 tipi di attacco:
            self.num_attack_types = 3
        Allora per ogni nodo ci sono:
            3 azioni di difesa specifiche + 1 azione di rilevamento = 4 azioni totali per nodo
        Quindi:
            Nodo 0 → azioni 0, 1, 2, 3
            Nodo 1 → azioni 4, 5, 6, 7
            Nodo 2 → azioni 8, 9, 10, 11
        L’azione di rilevamento è sempre l’ultima del blocco.
        Quindi:
            detect_action_for(0) = 0 * 4 + 3 = 3
            detect_action_for(1) = 1 * 4 + 3 = 7
            detect_action_for(2) = 2 * 4 + 3 = 11
        """
        return node_id * (self.num_attack_types + 1) + self.num_attack_types


# Verifica rapida: si crea un'istanza e si controlla che le dimensioni tornino con quanto
# calcolato nella Sezione 2 (20 osservazioni, 20 azioni per la versione v21).
_test_env = DefenderGymWrapper(probe_env)
print(f"obs_dim = {_test_env.obs_dim}, n_actions = {_test_env.n_actions}, "
      f"nodo Data = {_test_env.data_node_id}")

# Funzione per gli algoritmi che non hanno bisogno di distinguere terminated da truncated
# (SARSA e DDQN)
def is_episode_done(terminated: bool, truncated: bool) -> bool:
    return bool(terminated or truncated)

## SEZIONE 4: MISURARE LE PERFORMANCE

In questo ambiente il reward è spesso nullo o poco informativo durante i singoli step, mentre il risultato vero emerge soprattutto a fine episodio. Per questo il reward cumulativo medio, da solo, non basta a capire la qualità del difensore, infatti per valutare correttamente l'agente il wrapper classifica l'esito leggendo lo stato vero del gioco (`state.hacked`/`state.detected`).

Per questo motivo si misura separatamente:

- **tasso di rilevamento** (`detected_rate`): quota di episodi in cui il difensore individua l'attaccante prima che raggiunga `Data`. E' la metrica di business piu' rilevante: e' letteralmente la percentuale di attacchi respinti.
- **tasso di violazione** (`breached_rate`): quota di episodi in cui l'attaccante arriva a `Data`. Nel contesto sanitario del progetto, questa e' la metrica che GreenGuard vuole minimizzare a ogni costo, perche' corrisponde a dati di pazienti compromessi.
- **tasso di timeout** (`timeout_rate`): quota di episodi che raggiungono il limite interno di **100 passi** (`MAX_GAME_STEPS`, verificato nel sorgente di `gym-idsgame`) senza che ne' `state.hacked` ne' `state.detected` siano diventati veri. Nelle esecuzioni di questo notebook risulta **sempre 0%**: la rete ha solo 4 nodi e il livello di rilevamento cresce rapidamente verso il tetto (fino al 90% di probabilita' per passo), quindi ogni episodio si risolve - con un rilevamento o una violazione - molto prima dei 100 passi. La teniamo comunque distinta e monitorata: resta un esito possibile in linea di principio, utile a scoprire se in futuro (rete piu' grande, difensore molto piu' debole) iniziasse a comparire.
- **reward medio** e **lunghezza media dell'episodio**, utili soprattutto come curve di apprendimento durante il training (la loro media mobile ci dice se l'agente sta migliorando anche quando i tassi sopra sono ancora rumorosi).

Sono stati aggiunti anche due difensori semplici come termine di paragone, cioè come **baseline**.

Il primo è un **difensore casuale**, che a ogni passo sceglie un’azione a caso. Serve come riferimento minimo: un agente addestrato, come SARSA o DDQN, dovrebbe ottenere risultati migliori di una strategia completamente casuale. Se non ci riesce, significa probabilmente che l’addestramento non sta funzionando o che l’agente non sta imparando una policy utile.

Il secondo è un **difensore a regola fissa**, cioè un difensore euristico scritto a mano. In questo caso la regola è molto semplice: aumentare sempre il livello di rilevamento del nodo `Data`, ignorando il resto della rete. Questa strategia non è intelligente né adattiva, ma può comunque essere efficace se il nodo `Data` è particolarmente importante per l’attaccante.

Confrontare SARSA e DDQN con questa regola fissa serve a capire se gli agenti RL riescono non solo a fare meglio del caso, ma anche a superare una strategia manuale ragionevole. Se non riescono a batterla, non significa per forza che gli algoritmi siano inutili, ma indica che in questo scenario specifico l’euristica sul nodo `Data` è già molto forte oppure che l’addestramento degli agenti deve essere migliorato.


In [ ]:
# ============================================================
# FUNZIONI DI VALUTAZIONE CONDIVISE
# ============================================================
def evaluate_policy(env: DefenderGymWrapper, policy_fn, n_episodes: int = 300) -> dict:
    '''
    Esegue n_episodes episodi con la politica passata (una funzione obs -> azione) e
    restituisce un dizionario con le metriche descritte nella cella markdown sopra.
    Usata sia per gli agenti addestrati (in modalita' greedy) sia per le baseline ingenue.
    '''
    # Tre liste parallele, una entrata per episodio: 
    # - outcomes tiene traccia di come e' finita ogni partita ("detected"/"breached"/"timeout"), 
    # - rewards e lengths servono per le medie finali (avg_reward, avg_length).
    outcomes, rewards, lengths = [], [], []

    for _ in range(n_episodes):
        obs = env.reset()
        done = False
        ep_reward, ep_len = 0.0, 0

        # Un episodio dura finche' il wrapper non segnala che la partita e' finita
        # (rilevamento, violazione o timeout, vedi DefenderGymWrapper in Sezione 3).
        while not done:
            # policy_fn e' la vera "politica" da valutare: puo' essere una funzione che
            # campiona a caso (baseline), una regola fissa (l'euristica), o il metodo
            # choose_action di un agente addestrato chiamato in modalita' greedy. 
            action = policy_fn(obs)
            obs, reward, terminated, truncated, outcome = env.step(action)
            done = is_episode_done(terminated, truncated)
            ep_reward += reward  # accumuliamo il reward passo dopo passo
            ep_len += 1          # e contiamo quanti passi dura l'episodio

        # A episodio finito, registriamo i tre risultati in cui siamo interessati.
        outcomes.append(outcome)
        rewards.append(ep_reward)
        lengths.append(ep_len)

    # Counter conta quante volte compare ogni valore distinto nella lista: ci da' quindi
    # quanti episodi sono finiti "detected", quanti "breached", quanti "timeout" in un colpo solo.
    counts = Counter(outcomes)
    n = len(outcomes)

    # counts.get("detected", 0) restituisce 0 se quella chiave non compare mai:
    # senza il valore di default un KeyError interromperebbe il calcolo del tasso.
    return {
        "detected_rate": counts.get("detected", 0) / n,
        "breached_rate": counts.get("breached", 0) / n,
        "timeout_rate": counts.get("timeout", 0) / n,
        "avg_reward": float(np.mean(rewards)),
        "avg_length": float(np.mean(lengths)),
        "counts": dict(counts),  # utile per stampare i conteggi grezzi, non solo le percentuali
    }


def moving_average(values, window=100):
    '''Media mobile semplice, per leggere la tendenza di una curva rumorosa.'''
    if len(values) < window:
        return np.array(values, dtype=float)
    return np.convolve(values, np.ones(window) / window, mode="valid")


def print_metrics(label: str, metrics: dict):
    print(f"{label:22s} | rilevati: {metrics['detected_rate']:6.1%} | "
          f"violazioni: {metrics['breached_rate']:6.1%} | "
          f"timeout: {metrics['timeout_rate']:6.1%} | "
          f"reward medio: {metrics['avg_reward']:6.3f} | "
          f"durata media: {metrics['avg_length']:5.2f} passi")

In [ ]:
# ============================================================
# BASELINE SU idsgame-random_attack-v21
# ============================================================
env_random_attack = DefenderGymWrapper(probe_env)

# Questa politica risposponde con un'azione casuale
random_defender_policy = lambda obs: env_random_attack.env.defender_action_space.sample()
# Questa politica risponde sempre con la stessa identica azione, calcolata da detect_action_for(data_node_id).
# Quindi la stessa mossa viene ripetuta meccanicamente a ogni passo, di ogni episodio, senza eccezioni.
always_detect_data_policy = lambda obs: env_random_attack.detect_action_for(env_random_attack.data_node_id)

baseline_random_ra = evaluate_policy(env_random_attack, random_defender_policy, n_episodes=300)
baseline_heuristic_ra = evaluate_policy(env_random_attack, always_detect_data_policy, n_episodes=300)

print("Baseline su idsgame-random_attack-v21:")
print_metrics("Difensore casuale", baseline_random_ra)
print_metrics("Sempre-rileva-Data", baseline_heuristic_ra)

Già da queste due baseline emerge un risultato interessante e in parte controintuitivo: la politica fissa che esegue sempre l’azione di rilevamento sul nodo Data ottiene prestazioni migliori rispetto al difensore casuale.

Il motivo diventa chiaro osservando il comportamento delle due strategie. La policy always_detect_data concentra continuamente l’azione di difesa sul nodo Data, aumentando rapidamente il suo livello di rilevamento fino al massimo. La policy casuale, invece, distribuisce le azioni tra più nodi e quindi tende a disperdere le risorse difensive.

In uno scenario con un solo nodo Server intermedio, proteggere in modo costante il nodo Data, che rappresenta il punto finale più critico, può risultare più efficace rispetto a una strategia non mirata.
> Per questo motivo, la policy always_detect_data rappresenta una baseline più forte e più utile per valutare le prestazioni del DQN rispetto alla policy casuale.

# PARTE 1: AGENTE SARSA

## SEZIONE 5: PERCHE' UNA Q-TABLE A DIZIONARIO

SARSA e' un algoritmo *tabellare*: associa a ogni stato un insieme di valori Q, uno per ciascuna azione possibile, e aggiorna questi valori durante il training.

In questo ambiente lo spazio degli stati teorico e' enorme: ogni stato e' descritto da 20 valori interi compresi tra 0 e 9, quindi il numero massimo di combinazioni possibili e' 10^20. Creare in anticipo una Q-table con una riga per ogni possibile stato sarebbe quindi impraticabile, oltre che inutile, perche' molti di quegli stati potrebbero non essere mai visitati.

Per questo la Q-table viene implementata come un **dizionario**. Ogni stato realmente incontrato durante il training viene usato come chiave del dizionario, mentre il valore associato e' il vettore dei Q-value delle azioni disponibili. Se uno stato compare per la prima volta, la relativa voce viene creata al momento e inizializzata a zero e aggiornata ogni qualvolta venga eseguita.

Esempio:

```python
Q = {
    (0, 1, 0, 3, 2, 0, 1, 4, 0, 2, 1, 0, 3, 0, 1, 2, 0, 0, 1, 3):
        [0.0, 0.0, 0.5, 0.0, 1.2, 0.0, 0.0, 0.3, 0.0, 0.0,
         0.0, 0.0, 0.0, 0.7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
}
```

> **Il dizionario permette di memorizzare solo gli stati effettivamente visitati, evitando di creare una Q-table enorme con stati che probabilmente non verranno mai incontrati.**

In [ ]:
# ============================================================
# AGENTE SARSA (Q-table a dizionario)
# ============================================================
class SarsaAgent:
    '''
    Implementazione tabellare di SARSA con esplorazione epsilon-greedy ed epsilon
    decrescente nel tempo.

    Scelte di progettazione:
    - Q-table come defaultdict(np.zeros): ogni stato mai visto prima viene inizializzato
      a zero automaticamente al primo accesso.
    - la chiave della tabella e' la tupla dei 20 interi dell'osservazione: essendo valori
      discreti (0-9), non serve nessuna discretizzazione aggiuntiva.
    - update SARSA "puro": la stima del prossimo stato usa il valore dell'azione che
      l'agente sceglierebbe DAVVERO al passo successivo (next_action, gia' campionata con
      la stessa politica epsilon-greedy), non il massimo assoluto come farebbe Q-learning.
    '''

    def __init__(self, n_actions: int, alpha: float = 0.1, gamma: float = 0.95,
                 epsilon: float = 1.0, epsilon_min: float = 0.05, epsilon_decay: float = 0.999):
        self.n_actions = n_actions
        self.alpha = alpha          # tasso di apprendimento
        self.gamma = gamma          # fattore di sconto
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.q_table = defaultdict(lambda: np.zeros(n_actions, dtype=np.float32))

    def _key(self, obs: np.ndarray):
        """
        Converte l'osservazione in una chiave valida per la Q-table.

        L'osservazione viene convertita prima in interi e poi in una tupla,
        così da ottenere una chiave hashable utilizzabile nel dizionario
        che rappresenta la Q-table.

        Args:
            obs (np.ndarray): Osservazione corrente dell'ambiente.

        Returns:
            tuple: Rappresentazione hashable dello stato corrente.
        """
        return tuple(obs.astype(int))

    def choose_action(self, obs: np.ndarray, greedy: bool = False) -> int:
        """
        Seleziona un'azione a partire dallo stato corrente.

        Durante il training viene utilizata una strategia epsilon-greedy:
        - Se greedy=True, l'esplorazione viene disabilitata e viene valutato 
        in base al valore di epsilon se serve fare exploration o exploitation.
        Solo per evaluation si fa solo exploitatio, in quanto greedy viene passato a True.

        Args:
            obs (np.ndarray): Osservazione che rappresenta lo stato corrente.
            greedy (bool): Se True, seleziona sempre l'azione greedy,
                ignorando epsilon. Default: False.

        Returns:
            int: Indice dell'azione selezionata.
        """
        if not greedy and np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.q_table[self._key(obs)]))

    def update(self, obs, action, reward, next_obs, next_action, done):
        state, next_state = self._key(obs), self._key(next_obs)
        # Se l'episodio e' finito non c'e' un "prossimo valore" da propagare
        td_target = reward if done else reward + self.gamma * self.q_table[next_state][next_action]
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

## SEZIONE 6: TRAINING

I parametri scelti (`alpha=0.1`, `gamma=0.95`, `epsilon` che parte da 1.0 e decade moltiplicativamente dello 0.999 a ogni episodio fino a un minimo di 0.05) sono valori tipici per un problema tabellare di piccole dimensioni: un `alpha` non troppo piccolo permette di aggiornare velocemente stime basate su poche visite, mentre un decadimento di epsilon graduale su 5000 episodi lascia abbastanza tempo per esplorare prima di sfruttare quanto imparato.

Durante l'intero ciclo di training l'azione viene sempre scelta con `choose_action(obs)` (quindi sempre epsilon-greedy, mai `greedy=True`): SARSA e' un algoritmo *on-policy*, cioe' aggiorna la Q-table usando il valore dell'azione che l'agente sceglierebbe davvero al passo successivo (`next_action`, campionata con la stessa politica epsilon-greedy), non il massimo assoluto come farebbe Q-learning. Questo significa che, nelle prime fasi del training (epsilon alto), i valori appresi rispecchiano ancora il comportamento di una politica quasi casuale; solo con il decadere di epsilon i Q-value convergono verso quelli della politica quasi-greedy che si vuole ottenere - per questo e' importante lasciare abbastanza episodi prima che epsilon raggiunga il minimo.


In [ ]:
# ============================================================
# CICLO DI TRAINING SARSA
# ============================================================
def train_sarsa(env: DefenderGymWrapper, agent: SarsaAgent, n_episodes: int,
                starting_episode: int = 0) -> list:
    # starting_episode serve solo per il log qui sotto: quando si allena a blocchi
    # (vedi la chiamata piu' sotto, con i checkpoint), permette di stampare il numero
    # di episodio CUMULATIVO invece che ricominciare sempre da 1 a ogni blocco.
    history = []
    for episode in range(n_episodes):
        obs = env.reset()

        # Scelgo l'azione casuale nel processo di training sempre!
        action = agent.choose_action(obs)
        done = False
        ep_reward, ep_len, outcome = 0.0, 0, None

        while not done:
            # Esecuzione dell'azione
            next_obs, reward, terminated, truncated, outcome = env.step(action)
            done = is_episode_done(terminated, truncated)

            # SARSA ha bisogno della PROSSIMA azione prima di poter aggiornare la tabella:
            # la si campiona subito, con la stessa politica epsilon-greedy usata per agire
            next_action = agent.choose_action(next_obs)

            # --- UPDATE
            agent.update(obs, action, reward, next_obs, next_action, done)

            # prendo lo stato previsto e l'azione prevista
            obs, action = next_obs, next_action
            ep_reward += reward
            ep_len += 1

            if done:
                break

        # Si riduce epsilon per il prossimo episodio: con l'allenamento che avanza
        # si esplora sempre un po' meno.
        agent.decay_epsilon()
        history.append({"reward": ep_reward, "length": ep_len, "outcome": outcome, "epsilon": agent.epsilon})

        if (episode + 1) % 1000 == 0:
            recent = history[-1000:]
            det_rate = sum(h["outcome"] == "detected" for h in recent) / len(recent)
            breached_rate = sum(h["outcome"] == "breached" for h in recent) / len(recent)
            timeout_rate = sum(h["outcome"] == "timeout" for h in recent) / len(recent)
            print(f"episodio {starting_episode + episode + 1:5d} | epsilon={agent.epsilon:.3f} | "
                  f"tasso di rilevamento (ultimi 1000 ep): {det_rate:.1%} | "
                  f"tasso di violazione (ultimi 1000 ep): {breached_rate:.1%} | "
                  f"tasso di timeout (ultimi 1000 ep): {timeout_rate:.1%}")
    return history


# Allenamento a blocchi (invece che con un'unica chiamata da 5000 episodi) per poter
# valutare la policy. Serve a distinguere un miglioramento reale da
# un singolo campione fortunato o sfortunato: una sola valutazione a fine training (la
# cella piu' sotto) non basta a saperlo.
sarsa_agent = SarsaAgent(n_actions=env_random_attack.n_actions)
checkpoints = [1000, 2000, 3000, 4000, 5000]
sarsa_checkpoint_evals = {}
sarsa_history = []
episodes_done = 0
t0 = time.time()
for checkpoint in checkpoints:
    sarsa_history += train_sarsa(env_random_attack, sarsa_agent,
                                  n_episodes=checkpoint - episodes_done,
                                  starting_episode=episodes_done)
    episodes_done = checkpoint

    # Valutazione pulita della policy COSI' COM'E' in questo momento del training (150
    # episodi invece dei 300 usati per il confronto ufficiale con le baseline: qui serve
    # solo a vedere l'andamento nel tempo, non un numero definitivo da citare da solo).
    sarsa_checkpoint_evals[checkpoint] = evaluate_policy(
        env_random_attack, lambda obs: sarsa_agent.choose_action(obs, greedy=True), n_episodes=150
    )
    detected = sarsa_checkpoint_evals[checkpoint]["detected_rate"]
    breached = sarsa_checkpoint_evals[checkpoint]["breached_rate"]
    print(f"--> dopo {checkpoint:5d} episodi di training, valutazione greedy (150 ep): "
          f"rilevati={detected:.1%} | violazioni={breached:.1%}")

print(f"\nTraining completato in {time.time()-t0:.1f} secondi.")
print(f"Stati distinti visitati nella Q-table: {len(sarsa_agent.q_table)}")

**Perche' una valutazione a piu' checkpoint.** 

Durante il training, ogni 1000 episodi vengono riportati il tasso di rilevamento, il tasso di violazione e il valore corrente di epsilon. Questi valori descrivono il comportamento dell’agente mentre sta ancora imparando e utilizzando la strategia ε-greedy.

Anche quando epsilon raggiunge il valore minimo di 0.05, l’esplorazione non scompare completamente: ad ogni passo rimane infatti una probabilità del 5% di scegliere un’azione casuale. Di conseguenza, i risultati osservati durante il training contengono ancora una piccola componente di rumore dovuta all’esplorazione.

Per distinguere questo effetto dalla qualità effettiva della policy appresa, ogni 1000 episodi viene eseguita anche una valutazione separata su 150 episodi con greedy=True. In questa fase l’agente non esplora e sceglie sempre l’azione con il Q-value più alto.

I risultati mostrano che anche le valutazioni greedy ai checkpoint oscillano in modo evidente: ad esempio il tasso di rilevamento passa da circa 54.7% dopo 1000 episodi, a 54.0% dopo 2000, scende a 41.3% dopo 3000, risale a 52.7% dopo 4000 e rimane intorno al 52.7% dopo 5000 episodi.

Questo è un risultato importante: le oscillazioni non dipendono soltanto dall’esplorazione presente durante il training, perché compaiono anche quando la policy viene valutata senza azioni casuali. Non emerge quindi, almeno in questa esecuzione, una tendenza stabile e progressiva al miglioramento della policy.

La Q-table continua comunque a crescere e raggiunge migliaia di stati distinti visitati, segno che l’agente sta incontrando molte configurazioni dell’ambiente. Tuttavia, in uno spazio degli stati molto ampio, visitare molti stati non significa necessariamente averli osservati abbastanza volte da stimare in modo affidabile i relativi Q-value.

In sintesi, il training mostra che SARSA continua ad apprendere e aggiornare la Q-table, ma le prestazioni della policy rimangono instabili. Le valutazioni greedy ai checkpoint sono quindi utili proprio per verificare se il comportamento osservato durante il training rappresenti un reale miglioramento della policy oppure solo una fluttuazione temporanea. In questo caso, i risultati suggeriscono che la policy non abbia ancora raggiunto una convergenza stabile.

In [ ]:
# ============================================================
# CURVE DI APPRENDIMENTO SARSA
# ============================================================
rewards = [h["reward"] for h in sarsa_history]
outcomes = [h["outcome"] for h in sarsa_history]
epsilons = [h["epsilon"] for h in sarsa_history]

detected_flags = [1.0 if o == "detected" else 0.0 for o in outcomes]
breached_flags = [1.0 if o == "breached" else 0.0 for o in outcomes]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].plot(moving_average(rewards, 200), color="#2196F3")
axes[0].set_title("Reward medio (media mobile, finestra 200 episodi)")
axes[0].set_xlabel("Episodio")
axes[0].set_ylabel("Reward difensore")

axes[1].plot(moving_average(detected_flags, 200), label="rilevati", color="#4CAF50")
axes[1].plot(moving_average(breached_flags, 200), label="violazioni", color="#E53935")
axes[1].set_title("Tasso di rilevamento vs violazione (media mobile)")
axes[1].set_xlabel("Episodio")
axes[1].set_ylabel("Quota episodi")
axes[1].legend()

axes[2].plot(epsilons, color="#9C27B0")
axes[2].set_title("Decadimento di epsilon")
axes[2].set_xlabel("Episodio")
axes[2].set_ylabel("epsilon")

plt.tight_layout()
plt.show()

**Lettura delle curve.** 

Il primo grafico mostra l’andamento del reward medio durante il training, calcolato tramite una media mobile su 200 episodi. Il reward presenta forti oscillazioni, alternando fasi di miglioramento e peggioramento. Non emerge però una crescita progressiva e stabile: anche nelle fasi finali del training il reward continua a variare sensibilmente. Questo suggerisce che le prestazioni dell’agente non si stanno stabilizzando verso un comportamento chiaramente migliore.

Il secondo grafico confronta il tasso di rilevamento (verde) con il tasso di violazione (rosso). Anche queste due metriche oscillano durante tutto il training e le curve continuano frequentemente a incrociarsi. In alcuni intervalli il tasso di rilevamento supera nettamente quello di violazione, ma il vantaggio non viene mantenuto nel tempo. Non emerge quindi una progressiva separazione delle due curve a favore dei rilevamenti.

Il terzo grafico mostra il decadimento di epsilon. All’inizio epsilon = 1, quindi l’agente esplora frequentemente scegliendo anche azioni casuali. Con il procedere del training epsilon diminuisce fino a raggiungere il valore minimo di 0.05 intorno ai 3000 episodi. Da quel momento l’agente sfrutta prevalentemente i valori appresi nella Q-table, mantenendo comunque una probabilità del 5% di esplorare a ogni scelta.

Nel complesso, le curve non mostrano una chiara convergenza delle prestazioni. Questo comportamento continua anche dopo che epsilon ha raggiunto il valore minimo, quindi le forti oscillazioni osservate non sembrano essere spiegabili solamente dall’elevata esplorazione delle prime fasi del training.

Questa interpretazione è inoltre coerente con le valutazioni greedy effettuate ai checkpoint, nelle quali l’esplorazione viene completamente eliminata: anche in quelle valutazioni il tasso di rilevamento varia sensibilmente tra diversi momenti del training. Le curve e i checkpoint suggeriscono quindi che, in questa esecuzione, SARSA non mostri una chiara convergenza verso una policy progressivamente migliore.

## SEZIONE 7: VALUTAZIONE FINALE E CONFRONTO CON LE BASELINE

In [ ]:
# ============================================================
# VALUTAZIONE GREEDY (epsilon=0) SU 300 EPISODI, COME PER LE BASELINE
# ============================================================
sarsa_eval = evaluate_policy(env_random_attack, lambda obs: sarsa_agent.choose_action(obs, greedy=True), n_episodes=300)

print("Confronto finale su idsgame-random_attack-v21 (300 episodi di valutazione, politica greedy):")
print_metrics("Difensore casuale", baseline_random_ra)
print_metrics("Sempre-rileva-Data", baseline_heuristic_ra)
print_metrics("SARSA (addestrato)", sarsa_eval)

**Valutazione finale e confronto con le baseline.**

Terminato il training, la policy SARSA viene valutata su 300 nuovi episodi utilizzando greedy=True: come spiegato sopra, in questa fase l’agente non effettua esplorazione, ma sceglie sempre l’azione con il Q-value più alto tra quelle apprese.

In questa esecuzione, SARSA ottiene un tasso di rilevamento del 53,3%, con il 46,7% di violazioni e un reward medio di -12,133. Il risultato è leggermente migliore in termini numerici rispetto al difensore casuale, che raggiunge il 51,7% di rilevamenti e un reward medio di -12,563, ma rimane nettamente inferiore alla baseline euristica Sempre-rileva-Data, che raggiunge il 77,0% di rilevamenti e un reward medio di -5,980.

SARSA non sembra quindi aver appreso, in questa esecuzione, una strategia significativamente migliore della scelta casuale, mentre rimane molto distante dalla semplice strategia euristica.

Come osservato durante il training, le prestazioni mostrano inoltre una notevole variabilità tra i diversi checkpoint. Per questo motivo, il 53,3% finale non deve essere interpretato come una misura definitiva delle prestazioni di SARSA, ma come il risultato della policy ottenuta in questa specifica esecuzione.

Al termine del training sono stati memorizzati 8354 stati distinti nella Q-table. Questo evidenzia una delle difficoltà dell'approccio tabellare: ogni stato viene memorizzato separatamente e ciò che viene appreso per uno stato non viene automaticamente trasferito ad altri stati simili.

Nel complesso, questo esperimento mostra alcune delle difficoltà che un approccio tabellare può incontrare in un ambiente con uno spazio degli stati molto ampio. Il confronto successivo con DDQN permetterà di verificare se l'utilizzo di una rete neurale, e quindi la possibilità di condividere l'informazione appresa tra osservazioni simili, possa portare a una policy più efficace e stabile.

# PARTE 2: AGENTE DOUBLE DQN
## SEZIONE 8: DA Q-TABLE A RETE NEURALE, E DA DQN A DOUBLE DQN

Il DQN "classico" stima Q(s,a) con una rete neurale invece che con una tabella, e la allena minimizzando la distanza tra la stima attuale e un target calcolato con l'equazione di Bellman:

```
target = reward + gamma * max_a' Q_target(s', a')
```

Il problema di questa formulazione e' che lo stesso `max` viene usato sia per **scegliere** quale sia la prossima azione migliore sia per **valutarne** il valore, con la stessa rete. Se la rete ha delle stime rumorose (ed e' cosi' quasi sempre, specialmente a inizio training), questo max tende a selezionare sistematicamente le azioni per cui la rete ha sovrastimato il valore per puro errore statistico, non perche' siano davvero le migliori: un bias di sovrastima che si accumula nel tempo.

**Double DQN** rompe questo accoppiamento usando due reti:

- la rete "online" (quella che alleniamo) **sceglie** quale azione sarebbe la migliore nello stato successivo;
- la rete "target" (una copia periodica della rete online, aggiornata piu' lentamente) **valuta** quanto vale quell'azione.

```
target = reward + gamma * Q_target(s', argmax_a' Q_online(s', a'))
```

Selezione e valutazione fatte da due reti diverse riducono la correlazione tra l'errore di stima e la scelta dell'azione, e quindi il bias di sovrastima. 

In [ ]:
# ============================================================
# REPLAY BUFFER
# ============================================================
# Uso namedtuple per dare un nome agli elementi della tupla, così non devo ricordarmi quale 
# valore si trova in ogni posizione.
# namedtuple viene infatti usato per creare una tupla i cui elementi hanno anche un nome.
Transition = namedtuple("Transition", ["obs", "action", "reward", "next_obs", "done"])


class ReplayBuffer:
    '''
    Memoria a capacita' fissa delle transizioni (obs, azione, reward, obs_successiva, done).

    Perche' serve: allenare la rete sulla transizione appena vissuta, una alla volta e nello
    stesso ordine in cui accade, produrrebbe batch fortemente correlati (esperienze
    consecutive di uno stesso episodio si assomigliano) e romperebbe l'assunzione di dati
    indipendenti su cui si basa la discesa del gradiente stocastica. Campionando batch
    casuali da un buffer che contiene esperienze di molti episodi diversi, rendiamo gli
    aggiornamenti molto piu' stabili.
    '''

    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size: int):
        # Converto direttamente in tensori PyTorch tutti i valori
        batch = random.sample(self.buffer, batch_size)
        obs = torch.tensor(np.array([t.obs for t in batch]), dtype=torch.float32)
        actions = torch.tensor([t.action for t in batch], dtype=torch.int64) 
        rewards = torch.tensor([t.reward for t in batch], dtype=torch.float32)
        next_obs = torch.tensor(np.array([t.next_obs for t in batch]), dtype=torch.float32)
        dones = torch.tensor([t.done for t in batch], dtype=torch.float32)
        return obs, actions, rewards, next_obs, dones

    def __len__(self):
        return len(self.buffer)

In [ ]:
env_random_attack.action_space.n

In [ ]:
# ============================================================
# RETE NEURALE (Q-network)
# ============================================================
class QNetwork(nn.Module):
    '''
    Multi-Layer Perceptron (MLP) utilizzato per approssimare la funzione Q.

    La rete riceve in input l'osservazione dello stato, rappresentata da `obs_dim`
    caratteristiche, e restituisce `n_actions` valori in output: un valore Q
    stimato per ciascuna azione possibile del difensore.

    La rete contiene due strati nascosti, ciascuno composto da `hidden_dim`
    neuroni (64 nella configurazione utilizzata), con funzione di attivazione ReLU.
    Sono stati scelti due strati nascosti relativamente piccoli invece di un
    singolo strato molto largo. Per un problema di dimensioni contenute non è
    necessario utilizzare una rete molto grande: una rete più larga aumenterebbe
    il numero di parametri e il costo computazionale senza garantire necessariamente
    prestazioni migliori.
    '''

    def __init__(self, obs_dim: int, n_actions: int, hidden_dim: int = 64):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, n_actions)

    def forward(self, x):
        # Input shape: [N, 20]
        x = torch.relu(self.fc1(x)) # 20 input -> 64 neuroni - Output shape: [N, 64]
        x = torch.relu(self.fc2(x)) # 64 input -> 64 neuroni - Output shape: [N, 64]
        return self.fc3(x) # 64 input -> 20 Q-value - Output shape: [N, 20]

In [ ]:
# ============================================================
# AGENTE DOUBLE DQN
# ============================================================
class DDQNAgent:
    '''
    Agente Double DQN: rete online + rete target, replay buffer, esplorazione
    epsilon-greedy con decadimento.

    Iperparametri e motivazione delle scelte:
    - gamma=0.99: orizzonte lungo relativamente all'episodio (episodi brevi, pochi passi),
      valorizza comunque bene il segnale terminale che e' l'unica vera fonte di reward.
    - Adam con lr=1e-3: scelta di default robusta per problemi piccoli come questo.
    - SmoothL1Loss (Huber loss) invece di MSE: penalizza meno gli errori grandi, utile
      perche' durante uno step normale il reward resta vicino a 0, mentre nello step in
      cui l'episodio finisce salta a un valore di grandezza variabile e imprevedibile
      (schema `dense_rewards_v3`: es. ~0 per un rilevamento, un valore negativo come -26 
      per una violazione) - un outlier occasionale nel target che la MSE amplificherebbe 
      quadraticamente, distorcendo il gradiente.
    - target_update_every=200 (hard update): la rete target viene sincronizzata con quella
      online ogni 200 passi di training, non a ogni passo, cosi' il bersaglio della
      regressione resta stabile abbastanza a lungo da poter essere davvero inseguito.
    - le osservazioni vengono divise per 9 (il valore massimo di un attributo) prima di
      entrare in rete: portare gli input in un range [0,1] aiuta l'ottimizzazione rispetto a
      lavorare direttamente con interi fino a 9.
    '''

    def __init__(self, obs_dim: int, n_actions: int, hidden_dim: int = 64, gamma: float = 0.99,
                 lr: float = 1e-3, buffer_size: int = 20000, batch_size: int = 64,
                 epsilon: float = 1.0, epsilon_min: float = 0.05, epsilon_decay: float = 0.995,
                 target_update_every: int = 200, obs_scale: float = 9.0):
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.target_update_every = target_update_every
        self.obs_scale = obs_scale

        self.online_net = QNetwork(obs_dim, n_actions, hidden_dim).to(device)
        self.target_net = QNetwork(obs_dim, n_actions, hidden_dim).to(device)
        # uso gli stessi pesi di online_net per target_net, così le due reti partono identiche, 
        # invece che con due inizializzazioni casuali diverse
        self.target_net.load_state_dict(self.online_net.state_dict()) 

        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.loss_fn = nn.SmoothL1Loss()
        self.buffer = ReplayBuffer(buffer_size)
        self.train_steps = 0

    def _normalize(self, obs: np.ndarray) -> np.ndarray:
        return obs / self.obs_scale

    def choose_action(self, obs: np.ndarray, greedy: bool = False) -> int:
        if not greedy and np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        with torch.no_grad():
            obs_t = torch.tensor(self._normalize(obs), dtype=torch.float32).unsqueeze(0).to(device)
            q_values = self.online_net(obs_t)
            return int(torch.argmax(q_values, dim=1).item())

    def store(self, obs, action, reward, next_obs, done):
        self.buffer.push(self._normalize(obs), action, reward, self._normalize(next_obs), done)

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return None  # aspettiamo di avere abbastanza esperienza prima di allenare

        # Estraiamo casualmente un batch dal buffer con la funzione sample precedentemente scritta
        obs, actions, rewards, next_obs, dones = self.buffer.sample(self.batch_size)
        # Trasferiamo tutto su dispositivo scelto, CPU o GPU
        obs, actions, rewards = obs.to(device), actions.to(device), rewards.to(device)
        next_obs, dones = next_obs.to(device), dones.to(device)

        # Q(s,a) > q_values stimato dalla rete online per ogni azione possibile (scelte nel batch)
        q_values = self.online_net(obs).gather(1, actions.unsqueeze(1)).squeeze(1)

        # ---------------------------------------------------------
        # CALCOLO DEL TARGET ATTRAVERSO BELLMAN
        # ---------------------------------------------------------
        with torch.no_grad():
            # la rete ONLINE sceglie l'azione migliore nello stato successivo
            best_next_actions = torch.argmax(self.online_net(next_obs), dim=1)
            # la rete TARGET ne stimarne il valore
            next_q_target = self.target_net(next_obs).gather(1, best_next_actions.unsqueeze(1)).squeeze(1)
            # Calcoliamo il valore Q che la rete dovrebbe imparare.
            td_target = rewards + self.gamma * next_q_target * (1 - dones) # (1 - done) > serve per gestire gli stati terminali.

        # ---------------------------------------------------------
        # CALCOLO DELLA LOSS
        # ---------------------------------------------------------
        loss = self.loss_fn(q_values, td_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.train_steps += 1
        if self.train_steps % self.target_update_every == 0:
            # Aggiornamento "hard" della rete target:
            # ogni target_update_every passi copia i pesi della rete online nella rete target.
            # In questo modo il target rimane stabile per alcuni passi durante il training.
            self.target_net.load_state_dict(self.online_net.state_dict())

        return loss.item()

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

## SEZIONE 9: CICLO DI TRAINING (riutilizzabile per entrambi gli scenari)

Funzione di training che accetta un `DefenderGymWrapper` e un `DDQNAgent` gia' costruiti: la stessa funzione servira' sia per `random_attack` sia per `maximal_attack`, senza duplicare codice.

In [ ]:
# ============================================================
# CICLO DI TRAINING DDQN (condiviso tra i due scenari)
# ============================================================
def train_ddqn(env: DefenderGymWrapper, agent: DDQNAgent, n_episodes: int, log_every: int = 300,
               starting_episode: int = 0) -> list:
    # starting_episode serve solo per il log sotto: permette di stampare il numero di 
    # episodio CUMULATIVO invece di ricominciare sempre da capo ad ogni blocco.
    history = []
    for episode in range(n_episodes):
        obs = env.reset()
        done = False
        ep_reward, ep_len, outcome = 0.0, 0, None

        while not done:
            action = agent.choose_action(obs)
            next_obs, reward, terminated, truncated, outcome = env.step(action)
            done = is_episode_done(terminated, truncated)
            agent.store(obs, action, reward, next_obs, done)
            agent.train_step()
            obs = next_obs
            ep_reward += reward
            ep_len += 1

        # Riduzione graduale di epsilon
        agent.decay_epsilon()
        history.append({"reward": ep_reward, "length": ep_len, "outcome": outcome, "epsilon": agent.epsilon})

        if (episode + 1) % log_every == 0:
            recent = history[-log_every:]
            det_rate = sum(h["outcome"] == "detected" for h in recent) / len(recent)
            print(f"episodio {starting_episode + episode + 1:5d} | epsilon={agent.epsilon:.3f} | "
                  f"tasso di rilevamento (ultimi {log_every} ep): {det_rate:.1%}")
    return history

## SEZIONE 10: TRAINING SU idsgame-random_attack-v21

In [ ]:
# Si allena a blocchi (stessa idea di SARSA, Sezione 9), con un budget piu' ampio di
# quello originale (5000 episodi invece di 1500), per vedere se qui piu' episodi portano
# a un miglioramento piu' stabile - a differenza di quanto osservato con la Q-table
# tabellare di SARSA, una rete neurale generalizza tra osservazioni simili, quindi
# potrebbe sfruttare meglio un budget di training maggiore.
ddqn_agent_random = DDQNAgent(obs_dim=env_random_attack.obs_dim, n_actions=env_random_attack.n_actions)
checkpoints = [1000, 2000, 3000, 4000, 5000]
ddqn_history_random = []
episodes_done = 0
t0 = time.time()
for checkpoint in checkpoints:
    ddqn_history_random += train_ddqn(env_random_attack, ddqn_agent_random,
                                       n_episodes=checkpoint - episodes_done,
                                       starting_episode=episodes_done)
    episodes_done = checkpoint
    # Valutazione pulita (greedy=True, niente esplorazione) della policy in questo
    # momento del training - stesso motivo di SARSA: distinguere un miglioramento reale
    # da un singolo campione fortunato o sfortunato.
    eval_cp = evaluate_policy(env_random_attack, lambda obs: ddqn_agent_random.choose_action(obs, greedy=True), n_episodes=150)
    print(f"--> dopo {checkpoint:5d} episodi di training, valutazione greedy (150 ep): "
          f"rilevati={eval_cp['detected_rate']:.1%} | violazioni={eval_cp['breached_rate']:.1%}")
print(f"\nTraining completato in {time.time()-t0:.1f} secondi.")

In [ ]:
ddqn_eval_random = evaluate_policy(env_random_attack, lambda obs: ddqn_agent_random.choose_action(obs, greedy=True), n_episodes=300)

print("Confronto su idsgame-random_attack-v21 (300 episodi di valutazione, politica greedy):")
print_metrics("Difensore casuale", baseline_random_ra)
print_metrics("Sempre-rileva-Data", baseline_heuristic_ra)
print_metrics("SARSA (addestrato)", sarsa_eval)
print_metrics("DDQN (addestrato)", ddqn_eval_random)

**Lettura dei risultati su random_attack.**

Durante il training di DDQN il tasso di rilevamento presenta forti oscillazioni e non mostra una crescita progressiva e stabile. Le valutazioni greedy effettuate ogni 1000 episodi confermano questa variabilità: il tasso di rilevamento passa dal 44,7% dopo 1000 episodi al 56,7% dopo 2000, raggiunge il 64,0% dopo 3000, per poi scendere al 40,7% dopo 4000 e al 39,3% dopo 5000 episodi.

La valutazione finale su 300 nuovi episodi restituisce un tasso di rilevamento del 43,7%, con il 56,3% di violazioni e un reward medio di -14,647. In questa esecuzione DDQN risulta quindi inferiore sia al difensore casuale, che raggiunge il 51,7% di rilevamenti, sia alla baseline euristica Sempre-rileva-Data, che raggiunge il 77,0%.

Il risultato mostra che, nello scenario con attaccante casuale, DDQN non ha appreso in questa esecuzione una policy più efficace delle baseline. Inoltre, l'andamento dei checkpoint evidenzia che una buona prestazione osservata in una fase intermedia del training non viene necessariamente mantenuta fino alla fine.

## SEZIONE 11: TRAINING SU idsgame-maximal_attack-v21

L'attaccante ora e' deterministico e concentra sempre l'attacco sull'attributo del nodo raggiungibile con il valore piu' alto, invece di scegliere a caso. Ci
aspettiamo che questo cambi in modo sostanziale l'efficacia delle baseline ingenue (un difensore casuale ha meno probabilita' di "azzeccare" per caso il nodo giusto contro un attaccante che non si disperde), e vogliamo vedere se DDQN riesce ad imparare un pattern di difesa piu' efficace proprio perche' l'attaccante, essendo deterministico, e' in un certo senso piu'
prevedibile di uno casuale.

Usiamo la stessa identica configurazione (stessi iperparametri, stesso numero di episodi) usata per `random_attack`: cosi' il confronto tra i due scenari misura l'effetto della strategia dell'attaccante, non una differenza nella configurazione dell'agente.

In [ ]:
# ============================================================
# BASELINE SU idsgame-maximal_attack-v21
# ============================================================
env_maximal_attack = DefenderGymWrapper(IdsGameMaximalAttackV21Env())
env_maximal_attack.env.defender_action_space.seed(SEED)

random_defender_policy_ma = lambda obs: env_maximal_attack.env.defender_action_space.sample()
always_detect_data_policy_ma = lambda obs: env_maximal_attack.detect_action_for(env_maximal_attack.data_node_id)

baseline_random_ma = evaluate_policy(env_maximal_attack, random_defender_policy_ma, n_episodes=300)
baseline_heuristic_ma = evaluate_policy(env_maximal_attack, always_detect_data_policy_ma, n_episodes=300)

print("Baseline su idsgame-maximal_attack-v21:")
print_metrics("Difensore casuale", baseline_random_ma)
print_metrics("Sempre-rileva-Data", baseline_heuristic_ma)

Ed ecco la conferma dell'intuizione: il difensore casuale crolla rispetto allo scenario precedente, perche' un attaccante che concentra sempre lo sforzo sullo stesso punto debole sfrutta molto meglio le mosse sprecate a caso dal difensore. La regola fissa "sempre rileva Data" regge comunque bene (presidia il collo di bottiglia finale indipendentemente da come si comporta l'attaccante). Questo e' esattamente lo scenario in cui un agente che impara una strategia mirata, invece di applicarne una fissa o casuale, dovrebbe avere piu' spazio per emergere.

In [ ]:
# ============================================================
# TRAINING DDQN SU maximal_attack (stessi iperparametri di random_attack)
# ============================================================
# Stessa idea di random_attack: si allena a checkpoint invece che con una singola 
# chiamata, con una valutazione greedy pulita (150 episodi) ad ogni tappa.
ddqn_agent_maximal = DDQNAgent(obs_dim=env_maximal_attack.obs_dim, n_actions=env_maximal_attack.n_actions)
checkpoints = [1000, 2000, 3000, 4000, 5000]
ddqn_history_maximal = []
episodes_done = 0
t0 = time.time()
for checkpoint in checkpoints:
    ddqn_history_maximal += train_ddqn(env_maximal_attack, ddqn_agent_maximal,
                                        n_episodes=checkpoint - episodes_done,
                                        starting_episode=episodes_done)
    episodes_done = checkpoint
    eval_cp = evaluate_policy(env_maximal_attack, lambda obs: ddqn_agent_maximal.choose_action(obs, greedy=True), n_episodes=150)
    print(f"--> dopo {checkpoint:5d} episodi di training, valutazione greedy (150 ep): "
          f"rilevati={eval_cp['detected_rate']:.1%} | violazioni={eval_cp['breached_rate']:.1%}")
print(f"\nTraining completato in {time.time()-t0:.1f} secondi.")

In [ ]:
ddqn_eval_maximal = evaluate_policy(env_maximal_attack, lambda obs: ddqn_agent_maximal.choose_action(obs, greedy=True), n_episodes=300)

print("Confronto su idsgame-maximal_attack-v21 (300 episodi di valutazione, politica greedy):")
print_metrics("Difensore casuale", baseline_random_ma)
print_metrics("Sempre-rileva-Data", baseline_heuristic_ma)
print_metrics("DDQN (addestrato)", ddqn_eval_maximal)

**Lettura dei risultati su maximal_attack.**

Anche nello scenario maximal_attack l'andamento durante il training non è monotono: le valutazioni greedy ai checkpoint mostrano un tasso di rilevamento del 44,0% dopo 1000 episodi, 74,7% dopo 2000, 46,7% dopo 3000 e 66,0% dopo 4000 e 5000 episodi.

Le prestazioni continuano quindi a oscillare durante il training, ma il confronto finale con le baseline è più favorevole rispetto allo scenario random_attack.

Nella valutazione finale su 300 episodi, DDQN raggiunge un tasso di rilevamento del 65,3%, con il 34,7% di violazioni e un reward medio di -9,013. Il difensore casuale raggiunge invece soltanto il 35,0% di rilevamenti, mentre la policy euristica Sempre-rileva-Data arriva al 62,7%.

In questa esecuzione DDQN supera quindi nettamente la policy casuale e ottiene anche un risultato leggermente superiore alla baseline euristica. Il margine rispetto a Sempre-rileva-Data è tuttavia ridotto, pari a circa 2,6 punti percentuali, per cui non è sufficiente una singola esecuzione per concludere che DDQN sia stabilmente migliore dell'euristica.

Una possibile interpretazione è che l'attaccante maximal_attack, seguendo un comportamento più strutturato rispetto all'attaccante casuale, produca pattern più prevedibili che il DDQN può cercare di apprendere. Questa rimane però un'ipotesi: per verificarla sarebbe necessario ripetere l'esperimento con più seed e confrontare statisticamente i risultati.

## SEZIONE 12: CONFRONTO VISIVO TRA I DUE SCENARI

In [ ]:
# ============================================================
# CONFRONTO DELLE CURVE DI APPRENDIMENTO TRA I DUE SCENARI DDQN
# ============================================================
det_random = [1.0 if h["outcome"] == "detected" else 0.0 for h in ddqn_history_random]
det_maximal = [1.0 if h["outcome"] == "detected" else 0.0 for h in ddqn_history_maximal]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(moving_average(det_random, 100), label="random_attack", color="#2196F3")
axes[0].plot(moving_average(det_maximal, 100), label="maximal_attack", color="#E53935")
axes[0].set_title("DDQN — tasso di rilevamento durante il training")
axes[0].set_xlabel("Episodio")
axes[0].set_ylabel("Quota episodi 'detected' (media mobile)")
axes[0].legend()

scenarios = ["random_attack", "maximal_attack"]
random_vals = [baseline_random_ra["detected_rate"], baseline_random_ma["detected_rate"]]
heuristic_vals = [baseline_heuristic_ra["detected_rate"], baseline_heuristic_ma["detected_rate"]]
ddqn_vals = [ddqn_eval_random["detected_rate"], ddqn_eval_maximal["detected_rate"]]

x = np.arange(len(scenarios))
width = 0.25
axes[1].bar(x - width, random_vals, width, label="Difensore casuale", color="#90CAF9")
axes[1].bar(x, heuristic_vals, width, label="Sempre-rileva-Data", color="#FFB74D")
axes[1].bar(x + width, ddqn_vals, width, label="DDQN", color="#2196F3")
axes[1].set_xticks(x)
axes[1].set_xticklabels(scenarios)
axes[1].set_ylabel("Tasso di rilevamento")
axes[1].set_title("Confronto finale per scenario")
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## CONCLUSIONI FINALI

### Tabella riepilogativa

### Tabella riepilogativa

| Difensore | Scenario | Rilevati | Violazioni | Reward medio |
|---|---|---:|---:|---:|
| Difensore casuale | random_attack | 51.7% | 48.3% | -12.563 |
| Sempre-rileva-Data | random_attack | 77.0% | 23.0% | -5.980 |
| **SARSA (addestrato)** | random_attack | **53.3%** | 46.7% | -12.133 |
| **DDQN (addestrato)** | random_attack | **43.7%** | 56.3% | -14.647 |
| Difensore casuale | maximal_attack | 35.0% | 65.0% | -16.900 |
| Sempre-rileva-Data | maximal_attack | 62.7% | 37.3% | -9.707 |
| **DDQN (addestrato)** | maximal_attack | **65.3%** | **34.7%** | **-9.013** |

*I valori riportati corrispondono alla valutazione finale della presente esecuzione, effettuata su 300 episodi con policy greedy, quindi senza esplorazione epsilon-greedy. Durante lo sviluppo sono state osservate variazioni tra run differenti, soprattutto per SARSA e DDQN; per questo motivo questi valori descrivono questa specifica esecuzione e non devono essere interpretati come prestazioni definitive degli algoritmi.*

### Risultato finale

Sullo scenario **random_attack**, le prestazioni degli algoritmi di Reinforcement Learning risultano poco stabili. 
- SARSA raggiunge nella valutazione finale il 53,3% di rilevamenti, leggermente sopra il difensore casuale (51,7%),
- DDQN si ferma al 43,7%, risultando in questa esecuzione inferiore anche alla baseline casuale. 
Entrambi rimangono comunque molto distanti dalla semplice euristica Sempre-rileva-Data, che raggiunge il 77,0%.

Le oscillazioni osservate durante il training e le differenze riscontrate tra diverse esecuzioni indicano che, in questo scenario, né SARSA né DDQN mostrano una convergenza chiaramente stabile verso una strategia superiore alle baseline. In particolare, la maggiore complessità del DDQN non produce automaticamente prestazioni migliori rispetto all'approccio tabellare.

Sullo scenario **maximal_attack**, il risultato è invece più favorevole a DDQN. Il difensore casuale raggiunge solamente il 35,0% di rilevamenti, la policy Sempre-rileva-Data il 62,7%, mentre DDQN raggiunge il 65,3%. In questa esecuzione il modello neurale riesce quindi a superare nettamente la strategia casuale e leggermente anche l'euristica fissa.

Il confronto tra i due scenari suggerisce che la struttura del comportamento dell'attaccante possa influenzare significativamente la capacità dell'agente di apprendere una strategia utile. Un attaccante deterministico potrebbe offrire pattern più facilmente apprendibili rispetto a un attaccante che seleziona casualmente le proprie azioni. Tuttavia, questa interpretazione deve essere considerata un'ipotesi e andrebbe verificata attraverso più esecuzioni e seed differenti.

SARSA e DDQN rappresentano inoltre due approcci molto diversi. SARSA utilizza una Q-table e memorizza separatamente i valori associati agli stati visitati, mentre DDQN utilizza una rete neurale per approssimare la funzione Q. In un ambiente piccolo entrambi gli approcci sono applicabili, ma al crescere dello spazio degli stati la soluzione tabellare diventa rapidamente difficile da gestire. Una rete neurale permette invece di utilizzare una rappresentazione parametrica della funzione Q e può generalizzare l'informazione appresa tra osservazioni simili.

### Limiti riscontrati

- L'ambiente utilizzato rappresenta una rete molto piccola rispetto a una reale infrastruttura sanitaria. I risultati ottenuti non possono quindi essere trasferiti direttamente a reti con decine o centinaia di nodi. Sarebbe necessario verificare se gli stessi comportamenti si mantengano anche su topologie più complesse.

- Il reward fornisce gran parte dell'informazione più importante alla fine dell'episodio, quando si verifica un rilevamento oppure una violazione. Un reward shaping più informativo durante gli step intermedi potrebbe facilitare l'apprendimento, ma dovrebbe essere progettato con attenzione per evitare di influenzare artificialmente la policy.

- Le prestazioni di SARSA e DDQN hanno mostrato una certa variabilità tra esecuzioni diverse. Per ottenere risultati più affidabili sarebbe opportuno ripetere gli esperimenti con più seed e confrontare media e variabilità delle metriche.

- In questo lavoro l'attaccante utilizza una policy predefinita (`random_attack` o `maximal_attack`) e non modifica il proprio comportamento in risposta al difensore. Uno scenario più realistico potrebbe prevedere due agenti adattivi, attaccante e difensore, addestrati contemporaneamente.

## SEZIONE AGGIUNTIVA PER TESTARE L'USO DI NUOVI STRUMENTI: TorchRL E Tianshou

Questa sezione **non fa parte del progetto richiesto**.
Qui, solo per curiosita' ed esplorazione, ho provato a collegare l'ambiente a due librerie esterne molto usate nella ricerca in RL (`TorchRL` e `Tianshou`), per vedere se un'implementazione "professionale" di Double DQN arriva a risultati comparabili a quella scritta a mano.

### Perche' serve un nuovo adattatore

`TorchRL` e `Tianshou` non accettano un ambiente qualsiasi: richiedono che sia davvero una sottoclasse di `gymnasium.Env`, con `observation_space` e `action_space` dichiarati come oggetti `gymnasium.spaces`, e con `step()` che restituisce il formato standard a 5 valori (`obs, reward, terminated, truncated, info`).

`DefenderGymWrapper` **non** e' fatto cosi': e' una classe Python normale, non una sottoclasse di `gymnasium.Env`, e non dichiara `observation_space`/`action_space` come oggetti `gymnasium.spaces` — motivo per cui queste librerie la rifiutano, anche se dalla Sezione 3 il suo `step()` restituisce ormai gli stessi `(obs, reward, terminated, truncated, esito)` dello standard (una precisione in piu' aggiunta successivamente, non presente nella prima versione di questa appendice). 

Per questo è necessario scrivere un adattatore **nuovo e separato**, che usa `DefenderGymWrapper` rendendolo compatibile con lo standard Gymnasium.


### Utlizzo del Collector

Nel codice SARSA e DDQN scritto manualmente, il ciclo di training gestisce direttamente sia la raccolta delle esperienze sia l’aggiornamento dell’agente. All’interno dello stesso ciclo vengono infatti eseguiti env.step(), la memorizzazione della transizione e l’aggiornamento dei valori o della rete.

Librerie come TorchRL e Tianshou, invece, separano esplicitamente queste due responsabilità. Per la raccolta delle esperienze utilizzano un oggetto dedicato, chiamato Collector, il cui compito è far interagire la policy con l’ambiente e raccogliere le transizioni, eventualmente inserendole in un replay buffer.

L’aggiornamento della rete viene poi gestito separatamente.


In [ ]:
# ============================================================
# ADATTATORE: DefenderGymEnv (solo per questa appendice)
# ============================================================
class DefenderGymEnv(gym.Env):
    '''
    Uso il DefenderGymWrapper gia' costruito esponendolo come un vero ambiente 
    gymnasium.Env, cosi' com'e' richiesto da librerie come TorchRL e Tianshou.
    DefenderGymEnv si occupa dello standard richiesto dalle librerie:
    - observation_space
    - action_space
    - reset() -> obs, info
    - step() -> obs, reward, terminated, truncated, info
    '''
    # attributo previsto dagli ambienti Gymnasium: serve a descrivere eventuali modalità di rendering.
    metadata = {"render_modes": []}

    def __init__(self, wrapper: DefenderGymWrapper):
        super().__init__()
        self.wrapper = wrapper
        # Dichiarazione dello spazio delle osservazioni:
        # Le librerie RL devono sapere prima di iniziare il training:
        # Le librerie RL devono sapere prima di iniziare il training:
        # - Che forma ha l’osservazione?  La lunghezza del vettore è wrapper.obs_dim.
        # - Che valori può contenere?     low=0.0, high=9.0 (dal max_value dell'ambiente)
        # - Che tipo numerico ha?         dtype=np.float32
        # - Quante azioni può scegliere l’agente? gym.spaces.Discrete(wrapper.n_actions)
        self.obs_scale = 9.0  # stesso valore usato da DDQNAgent, per confrontabilita'
        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0, shape=(wrapper.obs_dim,), dtype=np.float32
        )
        # Dichiaro lo spazio delle azioni: le azioni sono numeri interi da 0 a n-1.
        # wrapper.n_actions > azioni del difensore
        self.action_space = gym.spaces.Discrete(wrapper.n_actions)

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            super().reset(seed=seed)
        obs = self.wrapper.reset()
        return obs / self.obs_scale, {}

    def step(self, action):
        # Richiamo la funzione step di DefenderGymWrapper.
        obs, defender_reward, terminated, truncated, outcome = self.wrapper.step(int(action))
        # info resta SEMPRE vuoto:
        # provando Tianshou, un info con una chiave presente solo a fine episodio
        # (quindi diverso da passo a passo) corrompe in NaN il suo replay buffer. 
        # Un info sempre uguale evita il problema.
        return obs / self.obs_scale, defender_reward, terminated, truncated, {}


# Verifica: avvolgiamo l'ambiente random_attack gia' usato nelle Parti 1 e 2, senza
# ricrearlo, e controlliamo che l'adattatore si comporti come un ambiente gymnasium vero.
adapted_env = DefenderGymEnv(env_random_attack)
print("observation_space:", adapted_env.observation_space)
print("action_space:", adapted_env.action_space)

obs, info = adapted_env.reset()
print("reset() -> shape:", obs.shape, "| dentro observation_space?",
      adapted_env.observation_space.contains(obs))

# Test per adapted_env.
for i in range(3):
    a = adapted_env.action_space.sample()
    obs, reward, terminated, truncated, info = adapted_env.step(a)
    print(f"step {i}: action={a}, reward={reward}, terminated={terminated}, info={info}")
    if terminated or truncated:
        obs, info = adapted_env.reset()

# Uso check_env per controllare che  un ambiente rispetti tutte le regole richieste per essere 
# considerato un vero gymnasium.Env — non solo "gira senza errori" (come il test sopra), ma 
# cose tipo: le forme e i tipi di obs/reward/info sono quelli giusti, ogni osservazione 
# restituita sta davvero dentro observation_space, reset(seed=...) si comporta secondo le 
# convenzioni, ecc. 
# Il try/except serve perché so già che ci sarà un fallimento:
# rifacendo reset() con lo stesso seed e la stessa azione, l'ambiente NON da sempre lo stesso risultato.
# gym-idsgame decide se un attacco riesce o viene rilevato chiamando np.random.rand() direttamente, 
# senza legarlo al seed passato a reset(). LIMITE DELLA LIBRERIA.
from gymnasium.utils.env_checker import check_env

try:
    check_env(adapted_env, skip_render_check=True)
    print("check_env: superato senza eccezioni.")
except AssertionError as e:
    if "Deterministic" in str(e):
        print("check_env: tutto conforme TRANNE il controllo di determinismo, che qui")
        print("non puo' passare per il motivo spiegato sopra (randomness interna di")
        print("gym_idsgame non legata al seed di reset). Non e' un problema dell'adattatore.")
    else:
        raise

### Installazione delle librerie

Per i due test seguenti serve creare piu' istanze indipendenti dell'ambiente. 

In [ ]:
# Installazione delle due librerie esterne.
!pip install -q tianshou torchrl

def make_defender_gym_env():
    '''Crea una nuova istanza indipendente dell'ambiente random_attack, gia' adattata.'''
    # Un ambiente gym-idsgame completamente nuovo: stato interno (nodi, difese,
    # posizione dell'attaccante) separato da probe_env/env_random_attack.
    raw_env = IdsGameRandomAttackV21Env()
    wrapper = DefenderGymWrapper(raw_env)
    # E' l'adattatore: lo rende un vero gymnasium.Env, compatibile
    # con Tianshou e TorchRL (che altrimenti lo rifiuterebbero).
    return DefenderGymEnv(wrapper)

### Prova con Tianshou (Double DQN)

Si usa l'API "procedurale" di Tianshou: una rete, una policy, l'algoritmo `DQN` (che di default usa gia' `is_double=True`, cioe' Double DQN — lo si rende comunque esplicito), un collettore di dati e un training breve e dimostrativo (25000 passi in totale, non un addestramento completo come nelle Parti 1-2).

In [ ]:
import tianshou as ts
from tianshou.algorithm import DQN
# DiscreteQLearningPolicy: la "policy" — decide quale azione scegliere data un'osservazione,
# usando la rete e una regola epsilon-greedy (lo stesso concetto di choose_action).
from tianshou.algorithm.modelfree.dqn import DiscreteQLearningPolicy
# AdamOptimizerFactory: costruisce l'ottimizzatore Adam (stesso di self.optimizer in DDQNAgent).
from tianshou.algorithm.optim import AdamOptimizerFactory
# CollectStats: il tipo di statistiche che il "collettore" (poco sotto) restituisce a ogni giro.
from tianshou.data import CollectStats
# OffPolicyTrainerParams: la configurazione del ciclo di training (quanti passi, quanti epoch...).
from tianshou.trainer import OffPolicyTrainerParams
# Net: un MLP pronto all'uso, stesso ruolo della nostra classe QNetwork.
from tianshou.utils.net.common import Net
# SpaceInfo: legge comodamente le dimensioni di osservazione/azione da un ambiente gymnasium.
from tianshou.utils.space_info import SpaceInfo

# Creiamo più copie dell'ambiente per raccogliere esperienze da episodi diversi.
# DummyVectorEnv gestisce più ambienti insieme, ma li esegue comunque uno alla volta
# nello stesso processo: quindi non c'è vero parallelismo a livello di CPU.
# Questo permette però di ottenere esperienze più varie durante il training e il test.
num_training_envs, num_test_envs = 4, 4
training_envs = ts.env.DummyVectorEnv([make_defender_gym_env for _ in range(num_training_envs)])
test_envs = ts.env.DummyVectorEnv([make_defender_gym_env for _ in range(num_test_envs)])

# Un ambiente "usa e getta", solo per leggere le dimensioni di osservazione (20) e azione (20).
env0 = make_defender_gym_env()
space_info = SpaceInfo.from_env(env0)
# La rete: 20 input, due strati nascosti da 64, 20 output — stessa forma della nostra QNetwork.
net = Net(state_shape=space_info.observation_info.obs_shape,
          action_shape=space_info.action_info.action_shape, hidden_sizes=[64, 64])

# La policy: sceglie l'azione con il valore Q piu' alto, tranne che con probabilita'
# eps_training=0.3 durante il training (esplorazione, come il nostro epsilon) e
# eps_inference=0.0 in valutazione (sempre greedy, come il nostro choose_action(greedy=True)).
policy = DiscreteQLearningPolicy(
    model=net, action_space=env0.action_space, eps_training=0.3, eps_inference=0.0,
)
# Configuriamo il DQN: qui definiamo come viene calcolato il target
# e ogni quanto viene aggiornata la rete target.
algorithm = DQN(
    policy=policy, optim=AdamOptimizerFactory(lr=1e-3), gamma=0.99,
    # Target TD a 1 passo (controlla su quanti passi si calcola il target TD):
    # usa reward + gamma * Q_target(stato_successivo).
    # Con 1 mantiene lo stesso approccio usato nella nostra DDQNAgent.
    n_step_return_horizon=1, 
    # Hard update della rete target:
    # ogni 200 passi copia i pesi della rete online nella rete target.
    target_update_freq=200,
    is_double=True,  # Double DQN esplicito
)

# Collettore usato durante il training:
# esegue la policy sui 4 ambienti, raccoglie le transizioni
# e le salva nel replay buffer per poterle usare negli aggiornamenti della rete 
# (come la classe ReplyBuffer).
training_collector = ts.data.Collector[CollectStats](
    algorithm, training_envs, ts.data.VectorReplayBuffer(20000, num_training_envs),
     # Mantiene attiva l'esplorazione epsilon-greedy durante il training.
    exploration_noise=True,  
)
# Collettore usato per la valutazione:
# esegue la policy sugli ambienti di test, ma non salva le transizioni
# in un replay buffer perché non deve allenare la rete.
test_collector = ts.data.Collector[CollectStats](algorithm, test_envs, exploration_noise=True)

# Allenamento a checkpoint (stessa idea di SARSA/DDQN), portando il budget totale a 25000
# passi - paragonabile ai ~5000 episodi * 4-5 passi/episodio usati per SARSA/DDQN. 
# show_progress=False evita di riempire l'output con le barre di avanzamento di Tianshou, 
# ripetute qui 5 volte invece di una.
checkpoints_passi = [5000, 10000, 15000, 20000, 25000]
# Epsilon decrescente sui checkpoint (0.3 -> 0.05, stessi estremi di SARSA/DDQN),
# invece di restare fisso al 30% per tutto il training come nella versione precedente:
# set_eps_training() e' il metodo di Tianshou per farlo (verificato leggendo il sorgente
# di DiscreteQLearningPolicy).
eps_per_checkpoint = [0.3 - i * (0.3 - 0.05) / (len(checkpoints_passi) - 1) for i in range(len(checkpoints_passi))]
steps_done = 0
t0 = time.time()
for cp, eps in zip(checkpoints_passi, eps_per_checkpoint):
    policy.set_eps_training(eps)
    result = algorithm.run_training(
        OffPolicyTrainerParams(
            training_collector=training_collector,
            test_collector=test_collector,
            max_epochs=1,
            epoch_num_steps=cp - steps_done,
            collection_step_num_env_steps=10,
            test_step_num_episodes=20,
            batch_size=64,
            update_step_num_gradient_steps_per_sample=1 / 10,
            test_in_training=False,
            show_progress=False,
        )
    )
    steps_done = cp
    # Valutazione pulita (greedy) della policy in questo momento del training - stesso
    # motivo di SARSA/DDQN: un singolo numero finale non basta a sapere se e' stabile.
    policy.eval()
    eval_cp = evaluate_policy(env_random_attack, lambda obs: policy.compute_action(obs), n_episodes=150)
    print(f"--> dopo {cp:5d} passi (eps_training={eps:.3f}), valutazione greedy (150 ep): "
          f"rilevati={eval_cp['detected_rate']:.1%} | violazioni={eval_cp['breached_rate']:.1%}")
    policy.train()  # si torna in modalita' training prima del prossimo blocco
print(f"\nTianshou: finito in {time.time()-t0:.1f}s totali.")

# L'output sopra (test_reward con media e deviazione standard) non e' nello stesso formato
# delle metriche usate per SARSA/DDQN: non si vede il tasso di rilevamento,
# solo un reward medio poco interpretabile da solo. Valutiamo quindi la policy addestrata
# con la STESSA evaluate_policy della Sezione 4, sullo STESSO env_random_attack usato per
# SARSA e DDQN per confrontare i risultati.

# -- VALUTAZIONE
policy.eval()  

# Equivalente alla valutazione greedy (policy.compute_action(obs) e' il 
# metodo di Tianshou equivalente choose_action):
# per ogni stato sceglie l'azione con Q-value più alto.
tianshou_eval = evaluate_policy(env_random_attack, lambda obs: policy.compute_action(obs), n_episodes=300)

print("\nConfronto su idsgame-random_attack-v21 (300 episodi, stesso formato di Sezione 4/7):")
print_metrics("Difensore casuale", baseline_random_ra)
print_metrics("Sempre-rileva-Data", baseline_heuristic_ra)
print_metrics("SARSA (addestrato)", sarsa_eval)
print_metrics("DDQN (addestrato)", ddqn_eval_random)
print_metrics("Tianshou DQN (addestrato)", tianshou_eval)

### Prova con TorchRL (DQNLoss, double_dqn=True)

TorchRL lavora a un livello un po' piu' basso: si costruisce la rete, si avvolge in un
`QValueActor`, si aggiunge un modulo di esplorazione epsilon-greedy, e si usa `DQNLoss` con
`double_dqn=True` per calcolare l'errore da minimizzare. Il ciclo di training qui e' scritto a
mano (raccogli un po' di dati, aggiorna la rete, ripeti), sempre a scopo dimostrativo.

In [ ]:
import torch.nn as nn
# GymWrapper: la stessa idea del nostro DefenderGymEnv, ma nella direzione opposta — prende
# un ambiente gymnasium.Env (il nostro, gia' adattato) e lo traduce nel formato di TorchRL
# (basato su TensorDict, un dizionario di tensori, non su tuple come gymnasium).
from torchrl.envs.libs.gym import GymWrapper
# StepCounter: un "transform" opzionale di TorchRL che aggiunge il conteggio dei passi
# all'osservazione. Non ci serve per la nostra logica, lo teniamo solo perche' e' la
# convenzione standard mostrata nella documentazione di TorchRL.
from torchrl.envs import TransformedEnv, StepCounter
# QValueActor: prende in input i Q-value prodotti dalla rete e sceglie l'azione con valore Q più alto.
# In pratica trasforma la rete neurale in una vera policy utilizzabile dall'agente.
# EGreedyModule: aggiunge l'esplorazione epsilon-greedy alla policy:
# con probabilità epsilon sceglie un'azione casuale, altrimenti usa quella con Q-value più alto.
from torchrl.modules import QValueActor, EGreedyModule
# DQNLoss: implementa la stessa equazione di Bellman con Double DQN scritta a mano in
# Sezione 8 (rete online che scegli l'azione, rete target che la valuta).
# HardUpdate: copia periodicamente i pesi nella rete target (come target_update_every).
from torchrl.objectives import DQNLoss, HardUpdate
# Collector: fa interagire la policy con l'ambiente e raccoglie le transizioni
# (stato, azione, reward, stato successivo, done).
# Stesso ruolo del ciclo "while not done" nei nostri train_sarsa/train_ddqn.
from torchrl.collectors import Collector
# TensorDictReplayBuffer + LazyTensorStorage: la memoria delle esperienze passate,
# stesso ruolo della classe ReplayBuffer, qui gia' pronta all'uso.
from torchrl.data import TensorDictReplayBuffer, LazyTensorStorage
# TensorDictSequential: permette di concatenare più moduli in sequenza.
# Nel nostro caso può essere usato per eseguire prima la policy
# e poi applicare l'esplorazione epsilon-greedy.
from tensordict.nn import TensorDictSequential

# Costruzione dell'ambiente
torchrl_env = TransformedEnv(GymWrapper(make_defender_gym_env()), StepCounter())
# TorchRL descrive le dimensioni con degli "spec" invece che con gymnasium.spaces diretti:
# stesso identico numero (20) che avevamo trovato in Sezione 2, letto in un altro formato.
n_obs = torchrl_env.observation_spec["observation"].shape[-1]
n_actions = torchrl_env.action_spec.shape[-1]

# La rete: stessa identica forma della nostra QNetwork (Sezione 8) — 20 input, due strati
# nascosti da 64, 20 output — ma scritta qui con nn.Sequential invece che come classe a parte.
net = nn.Sequential(
    nn.Linear(n_obs, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, n_actions),
)
# Trasformiamo la rete "grezza" in un attore che sa scegliere l'azione migliore.
qvalue_actor = QValueActor(net, in_keys=["observation"], spec=torchrl_env.action_spec)
# Il modulo di esplorazione: epsilon parte da 0.3 e scende linearmente fino a 0.05 in 5000 passi
greedy = EGreedyModule(spec=torchrl_env.action_spec, eps_init=0.3, eps_end=0.05, annealing_num_steps=5000)
# La policy usata per raccogliere dati: prima sceglie l'azione migliore (qvalue_actor),
# poi il modulo di esplorazione puo' sostituirla con una a caso (con probabilita' epsilon).
policy_explore = TensorDictSequential(qvalue_actor, greedy)

# La funzione di perdita Double DQN: 
# - double_dqn=True e' l'equivalente diretto della logica scritta a mano nel metodo train_step() di DDQNAgent. 
# - delay_value=True e' richiesto da TorchRL per usare una rete target separata (obbligatorio con double_dqn=True).
loss_module = DQNLoss(value_network=qvalue_actor, action_space=torchrl_env.action_spec,
                       double_dqn=True, delay_value=True)
# Copia i pesi nella rete target ogni 200 passi di training (come target_update_every).
target_updater = HardUpdate(loss_module, value_network_update_interval=200)
# Stesso ottimizzatore Adam, stesso learning rate delle Parti 1-2.
optim = torch.optim.Adam(loss_module.parameters(), lr=1e-3)
# Stesso ruolo della ReplayBuffer: batch_size=64, allineato a DDQNAgent.
replay_buffer = TensorDictReplayBuffer(storage=LazyTensorStorage(max_size=20000), batch_size=64)

from tensordict import TensorDict

def torchrl_policy_fn(obs):
    # Converte l'osservazione numpy in un TensorDict,
    # il formato di input utilizzato da TorchRL.
    td = TensorDict({"observation": torch.as_tensor(obs, dtype=torch.float32)}, batch_size=[])
    with torch.no_grad():  # non ci serve calcolare i gradienti, c'è solo valutazione
        # Usa solo qvalue_actor:
        # calcola i Q-value e sceglie sempre l'azione con valore Q più alto,
        # senza applicare esplorazione epsilon-greedy.
        out_td = qvalue_actor(td)
    # L'azione restituita è codificata one-hot.
    # argmax() individua la posizione dell'1 e la converte
    # nell'indice intero dell'azione scelta.
    return int(out_td["action"].argmax().item())


# Collector:
# esegue la policy nell'ambiente e raccoglie le transizioni a blocchi di 50 passi.
# Il training dura complessivamente 25.000 passi, così da avere abbastanza esperienza
# per confrontare il risultato con SARSA/DDQN e controllare la policy periodicamente.
collector = Collector(torchrl_env, policy_explore, frames_per_batch=50, total_frames=25000)

checkpoints_passi = [5000, 10000, 15000, 20000, 25000]
next_checkpoint = 0
total_loss, n_updates = 0.0, 0
t0 = time.time()
# Il ciclo di training scritto a mano: per ogni nuovo blocco di esperienza raccolto...
for i, batch in enumerate(collector):
    # ...lo aggiungiamo al buffer (stesso ruolo di agent.store() in Sezione 8)...
    replay_buffer.extend(batch.reshape(-1))
    if len(replay_buffer) >= 64:
        # ...campioniamo un batch a caso dal buffer (stesso ruolo di buffer.sample())...
        sample = replay_buffer.sample()
        # ...calcoliamo la perdita Double DQN su quel batch...
        loss = loss_module(sample)["loss"]
        # ...e aggiorniamo la rete con la discesa del gradiente, esattamente come in train_step().
        optim.zero_grad()
        loss.backward()
        optim.step()
        target_updater.step()  # eventuale sincronizzazione della rete target
        total_loss += loss.item()
        n_updates += 1

    steps_done = (i + 1) * 50
    if next_checkpoint < len(checkpoints_passi) and steps_done >= checkpoints_passi[next_checkpoint]:
        cp = checkpoints_passi[next_checkpoint]
        eval_cp = evaluate_policy(env_random_attack, torchrl_policy_fn, n_episodes=150)
        print(f"--> dopo {cp:5d} passi di training, valutazione greedy (150 ep): "
              f"rilevati={eval_cp['detected_rate']:.1%} | violazioni={eval_cp['breached_rate']:.1%}")
        next_checkpoint += 1

print(f"TorchRL: finito in {time.time()-t0:.1f}s, {n_updates} aggiornamenti, loss media {total_loss/max(n_updates,1):.3f}")
# Si chiude esplicitamente il Collector di TorchRL: shutdown() chiude l'ambiente
# sottostante e libera i buffer di rollout preallocati, invece di aspettare il garbage
# collector - e' convenzione documentata della libreria. 
# Il Collector di Tianshou non viene chiuso per il motivo opposto: 
# usa DummyVectorEnv, che non possiede risorse a livello di sistema operativo 
# (nessun sottoprocesso o pipe da terminare) - chiuderlo sarebbe solo estetico, non necessario.
collector.shutdown()

# Valutazione finale ufficiale (300 episodi, come per le baseline/SARSA/DDQN/Tianshou):
torchrl_eval = evaluate_policy(env_random_attack, torchrl_policy_fn, n_episodes=300)

print("\nConfronto su idsgame-random_attack-v21 (300 episodi, stesso formato di Sezione 4/7):")
print_metrics("Difensore casuale", baseline_random_ra)
print_metrics("Sempre-rileva-Data", baseline_heuristic_ra)
print_metrics("SARSA (addestrato)", sarsa_eval)
print_metrics("DDQN (addestrato)", ddqn_eval_random)
print_metrics("Tianshou DQN (addestrato)", tianshou_eval)
print_metrics("TorchRL DQN (addestrato)", torchrl_eval)

### Valutazione dei risultati con Tianshou e TorchRL

#### Tianshou

Nell'esecuzione osservata, il tasso di rilevamento ai checkpoint e': 78.7% (dopo 5000 passi),
80.0% (10000), 76.7% (15000), 76.7% (20000), 75.3% (25000). La policy e' gia' efficace fin dal
primo checkpoint e resta stabile per tutto il resto del training, senza cali evidenti. La
valutazione finale su 300 episodi da' un tasso di rilevamento del 76.0%, con il 24.0% di
violazioni e un reward medio di -6.240 - un risultato che supera nettamente il difensore casuale
(51.7%) e si avvicina molto alla baseline euristica "Sempre-rileva-Data" (77.0%), restandone appena
sotto (alcuni checkpoint intermedi, come quello dopo 10000 passi all'80.0%, l'avevano gia' superata).

#### TorchRL

Nell'esecuzione osservata, il tasso di rilevamento ai checkpoint e': 46.0% (dopo 5000 passi),
47.3% (10000), 48.7% (15000), 37.3% (20000), 46.7% (25000). Il risultato oscilla per tutto il
training senza una tendenza chiara, restando vicino o sotto il livello del difensore casuale. La
valutazione finale su 300 episodi da' un tasso di rilevamento del 43.3%, con il 56.7% di
violazioni e un reward medio di -14.733 - sotto sia il difensore casuale (51.7%) sia la baseline
euristica (77.0%), e vicino al risultato di DDQN nella stessa esecuzione (43.7%).

#### Confronto tra le implementazioni

I risultati finali sullo scenario random_attack in questa esecuzione sono:

| Metodo | Rilevati | Violazioni | Reward medio |
|---|---|---|---|
| Difensore casuale | 51.7% | 48.3% | -12.563 |
| Sempre-rileva-Data | 77.0% | 23.0% | -5.980 |
| SARSA | 53.3% | 46.7% | -12.133 |
| DDQN manuale | 43.7% | 56.3% | -14.647 |
| Tianshou DQN | 76.0% | 24.0% | -6.240 |
| TorchRL DQN | 43.3% | 56.7% | -14.733 |

In questa esecuzione Tianshou e' l'approccio che ottiene il risultato migliore, restando molto
vicino alla baseline euristica. TorchRL, invece, ottiene prestazioni simili alla DDQN manuale e
inferiori alle baseline.

La differenza tra Tianshou e TorchRL non deve pero' essere interpretata automaticamente come una
superiorita' generale di una libreria rispetto all'altra. Le due implementazioni possono differire
per diversi dettagli di training, come gestione del replay buffer, frequenza degli aggiornamenti,
esplorazione, inizializzazione della rete, modalita' di raccolta delle esperienze e gestione della
rete target - alcuni di questi (buffer, batch, esplorazione) sono gia' stati allineati tra le due
librerie in questo notebook proprio per rendere il confronto piu' onesto, ma non tutte le
differenze implementative sono eliminabili.